In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('/Users/krishnag02/Learning Phase/Jupyterfiles/food_delivery_orders_dataset.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 37 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   order_id                             50000 non-null  str    
 1   customer_id                          50000 non-null  str    
 2   restaurant_id                        50000 non-null  str    
 3   driver_id                            50000 non-null  str    
 4   order_timestamp                      50000 non-null  str    
 5   order_date                           50000 non-null  str    
 6   order_hour                           50000 non-null  int64  
 7   day_of_week                          50000 non-null  str    
 8   is_weekend                           50000 non-null  int64  
 9   city                                 50000 non-null  str    
 10  delivery_area                        50000 non-null  str    
 11  customer_age                         50

# 🧑‍💼 Client Requirement #1 — Business Overview

We have recently collected our food-delivery order data and want to understand the overall health of the platform before we start deeper analysis.

Please provide me with the following:

- How many total orders do we have?
- How many unique customers placed orders?
- How many unique restaurants are present?
- What percentage of orders were successfully completed?
- What percentage of orders were cancelled?

Please return the results in a clean summary format.

In [4]:
total_orders = df.shape[0]
Customers = df['customer_id'].nunique()
Restaurants = df['restaurant_id'].nunique()
Percentage_completed = (((df.groupby('order_status')['order_id'].size()['Completed'])/df.shape[0])*100)
Percentage_cancelled = (((df.groupby('order_status')['order_id'].size()['Cancelled'])/df.shape[0])*100)

In [5]:
print(f'Total Orders : {total_orders}\nNo of Customers : {Customers}\nNo of Restaurants : {Restaurants}\nCompleted_orders: {Percentage_completed}%\nCancelled_order : {Percentage_cancelled}%')

Total Orders : 50000
No of Customers : 4979
No of Restaurants : 250
Completed_orders: 98.264%
Cancelled_order : 1.736%


# 🧑‍💼 Client Requirement #2 — Customer Behavior

Thanks. The overall cancellation rate looks low, but I want to understand which customers are contributing most to our order volume.

For every customer, calculate:

- Total number of orders
- Number of completed orders
- Number of cancelled orders
- Total amount spent
- Average order value

Then identify the top 10 customers by total amount spent.

One important business rule:

Cancelled orders should NOT contribute to customer spending.

Return the result sorted from highest to lowest total spending.

In [6]:
temp = df.groupby(['customer_id','order_status'])[['order_id','order_total']].agg({'order_id':'size','order_total':['sum','mean']}).unstack()
temp.fillna(0,inplace=True)

order_id           order_total                                 
                  size                   sum                  mean           
order_status Cancelled Completed   Cancelled  Completed  Cancelled  Completed
customer_id                                                                  
CUS_00001         65.0    4445.0     2912.78  146326.36  44.812000  32.919316
CUS_00002         44.0    2021.0     1382.92   67110.38  31.430000  33.206522
CUS_00003         36.0    2102.0     1735.55   91792.49  48.209722  43.669120
CUS_00004         37.0    2282.0     1151.28   84788.20  31.115676  37.155215
CUS_00005         23.0    1483.0      880.42   61700.51  38.279130  41.605199
...                ...       ...         ...        ...        ...        ...
CUS_07491          0.0       1.0        0.00      15.47   0.000000  15.470000
CUS_07494          0.0       3.0        0.00      99.93   0.000000  33.310000
CUS_07495          0.0       1.0        0.00      10.53   0.000000  10.530000
CUS_07497          0.0       1.0        0.00      21.17   0.000000  21.170000
CUS_07498          0.0       1.0        0.00      30.17   0.000000  30.170000

[4979 rows x 6 columns]

In [7]:
client = pd.DataFrame()
client['Total_order'] = (temp['order_id']['size']['Cancelled']) + (temp['order_id']['size']['Completed'])
client['Completed_orders'] = temp['order_id']['size']['Completed']
client['Cancelled_orders'] = temp['order_id']['size']['Cancelled']
client['Total_spent'] = temp['order_total']['sum']['Completed']
client['Avg_order_value'] = temp['order_total']['mean']['Completed']
client.sort_values('Total_spent',ascending=False).head(10)

,Total_order,Completed_orders,Cancelled_orders,Total_spent,Avg_order_value
customer_id,,,,,
CUS_00001,4510.0,4445.0,65.0,146326.36,32.919316
CUS_00003,2138.0,2102.0,36.0,91792.49,43.669120
CUS_00004,2319.0,2282.0,37.0,84788.20,37.155215
CUS_00002,2065.0,2021.0,44.0,67110.38,33.206522
CUS_00005,1506.0,1483.0,23.0,61700.51,41.605199
CUS_00006,1297.0,1277.0,20.0,52645.75,41.226116
CUS_00007,1214.0,1189.0,25.0,43895.30,36.917830
CUS_00008,1102.0,1076.0,26.0,35322.44,32.827546
CUS_00009,823.0,817.0,6.0,34223.30,41.888984


# 🧑‍💼 Client Requirement #3 — Restaurant Performance

Now let's move to the restaurant side.

The business team wants to identify which restaurants are performing best, but they don't want cancellations to inflate revenue.

For each restaurant, calculate:

- total_orders
- completed_orders
- cancelled_orders
- total_revenue — Completed orders only
- avg_order_value — Completed orders only
- avg_customer_rating — Completed orders only

Then answer:

What are the top 10 restaurants by total revenue?

In [8]:
temp = df.groupby(['restaurant_id','order_status'])[['order_id','order_total','customer_rating']].agg({'order_id' : 'size', 'order_total' : ['sum','mean'], 'customer_rating' : 'mean'}).unstack()
temp.fillna(0,inplace=True)

order_id           order_total                         \
                   size                   sum                   mean   
order_status  Cancelled Completed   Cancelled  Completed   Cancelled   
restaurant_id                                                          
RES_0001          162.0    8675.0     5557.29  286626.46   34.304259   
RES_0002           52.0    3501.0     1755.16  120102.18   33.753077   
RES_0003           64.0    3903.0     1570.62   81038.98   24.540937   
RES_0004           62.0    4199.0     2159.26  141270.79   34.826774   
RES_0005           20.0    1344.0      604.81   44809.81   30.240500   
...                 ...       ...         ...        ...         ...   
RES_0246            0.0      14.0        0.00     569.34    0.000000   
RES_0247            0.0      48.0        0.00    2444.45    0.000000   
RES_0248            0.0      13.0        0.00     426.13    0.000000   
RES_0249            0.0      21.0        0.00    2490.29    0.000000   
RES_0250            1.0      28.0      223.83    2652.30  223.830000   

                          customer_rating            
                                     mean            
order_status    Completed       Cancelled Completed  
restaurant_id                                        
RES_0001        33.040514             0.0  4.286110  
RES_0002        34.305107             0.0  4.308312  
RES_0003        20.763254             0.0  4.340584  
RES_0004        33.643913             0.0  4.275923  
RES_0005        33.340632             0.0  4.291815  
...                   ...             ...       ...  
RES_0246        40.667143             0.0  4.135714  
RES_0247        50.926042             0.0  4.435417  
RES_0248        32.779231             0.0  4.330769  
RES_0249       118.585238             0.0  4.300000  
RES_0250        94.725000             0.0  4.478571  

[250 rows x 8 columns]

In [9]:
client3 = pd.DataFrame(index=temp.index)
client3['Total_orders'] = (temp['order_id']['size']['Cancelled'] + temp['order_id']['size']['Completed'])
client3['Completed_orders'] = temp['order_id']['size']['Completed']
client3['Cancelled_orders'] = temp['order_id']['size']['Cancelled']
client3['Total_revenue'] = temp['order_total']['sum']['Completed']
client3['Avg_order_value'] = temp['order_total']['mean']['Completed']
client3['Avg_customer_rating'] = temp['customer_rating']['mean']['Completed']
client3.sort_values('Total_revenue', ascending = False).head(10)

,Total_orders,Completed_orders,Cancelled_orders,Total_revenue,Avg_order_value,Avg_customer_rating
restaurant_id,,,,,,
RES_0001,8837.0,8675.0,162.0,286626.46,33.040514,4.286110
RES_0004,4261.0,4199.0,62.0,141270.79,33.643913,4.275923
RES_0002,3553.0,3501.0,52.0,120102.18,34.305107,4.308312
RES_0009,1621.0,1595.0,26.0,82140.23,51.498577,4.281693
RES_0003,3967.0,3903.0,64.0,81038.98,20.763254,4.340584
RES_0008,1265.0,1247.0,18.0,62351.98,50.001588,4.340176
RES_0006,1141.0,1127.0,14.0,58309.62,51.738793,4.292990
RES_0012,466.0,458.0,8.0,46851.94,102.296812,4.315721
RES_0005,1364.0,1344.0,20.0,44809.81,33.340632,4.291815


# 🧑‍💼 Client Requirement #4 — City Performance

Now management wants to understand which cities are driving the business.

For each city, calculate:

- total_orders
- completed_orders
- cancelled_orders
- total_revenue — Completed orders only
- avg_order_value — Completed orders only
- cancellation_rate — percentage of all orders that were cancelled
- avg_delivery_time — Completed orders only

Then answer:

Which 5 cities generate the highest total revenue?

Important business rules
Revenue → Completed only
AOV → Completed only
Delivery time → Completed only
Cancellation rate → Cancelled / Total orders × 100

In [10]:
temp = df.groupby(['city','order_status'])[['order_id','order_total','actual_delivery_time_minutes']].agg({'order_id':'size', 'order_total':['sum','mean'],'actual_delivery_time_minutes':'mean'}).unstack()
temp.fillna(0,inplace = True)

order_id           order_total                                   \
                  size                   sum                  mean              
order_status Cancelled Completed   Cancelled  Completed  Cancelled  Completed   
city                                                                            
City_A             361     19527    13530.13  717733.52  37.479584  36.755954   
City_B             258     14627    10246.89  601434.09  39.716628  41.118075   
City_C             172     10006     6921.53  340471.27  40.241453  34.026711   
City_D              77      4972     3599.93  214537.69  46.752338  43.149173   

             actual_delivery_time_minutes             
                                     mean             
order_status                    Cancelled  Completed  
city                                                  
City_A                                0.0  34.991294  
City_B                                0.0  36.008546  
City_C                                0.0  34.056666  
City_D                                0.0  35.471239

In [11]:
client4 = pd.DataFrame(index=temp.index) 
client4['Total_orders'] = (temp['order_id']['size']['Cancelled'] + temp['order_id']['size']['Completed'])
client4['Completed_orders'] = temp['order_id']['size']['Completed']
client4['Cancelled_orders'] = temp['order_id']['size']['Cancelled']
client4['Total_revenue'] = temp['order_total']['sum']['Completed']
client4['Avg_order_value'] = temp['order_total']['mean']['Completed']
client4['Cancellation_rate'] = (client4['Cancelled_orders']/client4['Total_orders'])*100
client4['Avg_delivery_time'] = temp['actual_delivery_time_minutes']['mean']['Completed']
client4.sort_values('Total_revenue',ascending=False)

,Total_orders,Completed_orders,Cancelled_orders,Total_revenue,Avg_order_value,Cancellation_rate,Avg_delivery_time
city,,,,,,,
City_A,19888,19527,361,717733.52,36.755954,1.815165,34.991294
City_B,14885,14627,258,601434.09,41.118075,1.733289,36.008546
City_C,10178,10006,172,340471.27,34.026711,1.689919,34.056666
City_D,5049,4972,77,214537.69,43.149173,1.525054,35.471239


# 🧑‍💼 Client Requirement #5 — Delivery Performance

Now the operations team wants to investigate late deliveries.

For each city, calculate:

- total_completed_orders
- late_orders
- late_delivery_rate
- avg_delivery_time
- avg_estimated_delivery_time
- avg_delay_minutes

Business definitions

Only Completed orders should be considered.

Late order:

late_delivery == True

Late delivery rate:

late_orders / total_completed_orders × 100

Delay minutes:

actual_delivery_time_minutes - estimated_delivery_time_minutes

Then answer:

Which 5 cities have the highest late-delivery rate, considering only cities with at least 500 completed orders?

This one is a little harder because now you need to create a derived metric (delay) and apply a minimum-volume business filter before ranking.

In [12]:
df['delay_minutes'] = df['actual_delivery_time_minutes']- df['estimated_delivery_time_minutes']
temp = df[df['order_status'] == 'Completed'].groupby('city')[['order_id','late_delivery','estimated_delivery_time_minutes','actual_delivery_time_minutes','delay_minutes']].agg({'order_id' : 'size','late_delivery' : 'sum','estimated_delivery_time_minutes' : 'mean','actual_delivery_time_minutes':'mean', 'delay_minutes' : 'mean'})
temp.rename(columns={'order_id':'Total_completed_orders','late_delivery':'Late_orders','estimated_delivery_time_minutes':'Avg_estimated_delivery_time','actual_delivery_time_minutes':'Avg_delivery_time', 'delay_minutes':'Avg_delay_minutes'},inplace=True)
temp['Late_delivery_rate'] = (temp['Late_orders']/temp['Total_completed_orders'])*100
temp[temp['Total_completed_orders']>=500].sort_values('Late_delivery_rate',ascending=False).head(5)

,Total_completed_orders,Late_orders,Avg_estimated_delivery_time,Avg_delivery_time,Avg_delay_minutes,Late_delivery_rate
city,,,,,,
City_B,14627,5586.0,31.488754,36.008546,4.519792,38.189649
City_A,19527,7341.0,30.613305,34.991294,4.377989,37.594100
City_D,4972,1727.0,31.528962,35.471239,3.942277,34.734513
City_C,10006,3314.0,30.382271,34.056666,3.674395,33.120128


# 🧑‍💼 Client Requirement #6 — Driver Performance

Operations now wants to evaluate delivery partner performance.

For each driver_id, calculate:

- total_deliveries — Completed orders only
- avg_delivery_time
- avg_estimated_delivery_time
- avg_delay_minutes
- late_deliveries
- late_delivery_rate
- avg_driver_rating
- avg_distance_km

Then identify the top 10 drivers using this business rule:

A driver must have at least 100 completed deliveries, and among those drivers, return the 10 drivers with the lowest late-delivery rate.

In [13]:
temp = df[df['order_status'] == 'Completed'].groupby('driver_id')[
    [
        'order_id',
        'actual_delivery_time_minutes',
        'estimated_delivery_time_minutes',
        'delay_minutes',
        'late_delivery',
        'delivery_partner_rating',
        'distance_km']
        ].agg({
            'order_id':'size',
            'actual_delivery_time_minutes':'mean',
            'estimated_delivery_time_minutes':'mean',
            'delay_minutes':'mean',
            'late_delivery':'sum',
            'delivery_partner_rating':'mean',
            'distance_km':'mean'})

In [14]:
temp.rename(columns={
    'order_id':'total_deliveries',
    'actual_delivery_time_minutes':'avg_delivery_time',
    'estimated_delivery_time_minutes':'avg_estimated_delivery_time',
    'delay_minutes':'avg_delay_minutes',
    'late_delivery':'late_deliveries',
    'delivery_partner_rating':'avg_driver_rating',
    'distance_km':'avg_distance_km'
    }, inplace=True
)

In [15]:
temp['late_delivery_rate'] = (temp['late_deliveries']/temp['total_deliveries'])*100

In [16]:
temp[temp['total_deliveries']>=100].sort_values('late_delivery_rate').head(10)

,total_deliveries,avg_delivery_time,avg_estimated_delivery_time,avg_delay_minutes,late_deliveries,avg_driver_rating,avg_distance_km,late_delivery_rate
driver_id,,,,,,,,
DRV_0022,276,33.931159,30.880435,3.050725,74.0,5.0,4.792391,26.811594
DRV_0031,280,34.739286,31.439286,3.300000,81.0,5.0,4.987143,28.928571
DRV_0052,274,32.817518,29.897810,2.919708,80.0,4.4,4.656934,29.197080
DRV_0180,270,34.874074,31.714815,3.159259,79.0,5.0,4.951111,29.259259
DRV_0155,280,33.407143,30.164286,3.242857,82.0,4.5,4.661786,29.285714
DRV_0109,289,34.671280,31.117647,3.553633,85.0,4.8,4.721107,29.411765
DRV_0172,192,34.364583,30.687500,3.677083,57.0,4.3,4.749479,29.687500
DRV_0113,235,34.625532,31.046809,3.578723,71.0,4.9,4.825532,30.212766
DRV_0039,287,34.710801,30.996516,3.714286,88.0,4.6,4.855749,30.662021


# 🚀 Client Requirement #7 — Customer Segmentation

Now the marketing team wants to identify high-value customers.

Using your customer-level analysis, classify every customer into one of these segments:

- VIP

    Total spent >= ₹50,000
    AND completed orders >= 100

- High Value

    Total spent >= ₹25,000
    AND completed orders >= 50

- Regular

    Everyone else who has at least one completed order

Then provide:

- Number of customers in each segment
- Total revenue generated by each segment
- Average order value of each segment
- Percentage of the customer base represented by each segment

Finally answer:

- Which customer segment generates the most revenue?

In [17]:
client

,Total_order,Completed_orders,Cancelled_orders,Total_spent,Avg_order_value
customer_id,,,,,
CUS_00001,4510.0,4445.0,65.0,146326.36,32.919316
CUS_00002,2065.0,2021.0,44.0,67110.38,33.206522
CUS_00003,2138.0,2102.0,36.0,91792.49,43.669120
CUS_00004,2319.0,2282.0,37.0,84788.20,37.155215
CUS_00005,1506.0,1483.0,23.0,61700.51,41.605199
...,...,...,...,...,...
CUS_07491,1.0,1.0,0.0,15.47,15.470000
CUS_07494,3.0,3.0,0.0,99.93,33.310000
CUS_07495,1.0,1.0,0.0,10.53,10.530000


In [18]:
condition = [
    (client['Total_spent'] >= 50000) & (client['Completed_orders'] >= 100),
    (client['Total_spent'] >= 25000) & (client['Completed_orders'] >= 50),
    (client['Completed_orders'] >= 1)]

choices = ['VIP','High Value','Regular']

client['segments'] = np.select(condition,choices,default='Unknown')

In [19]:
client7 = pd.DataFrame()

client7['customers'] = client['segments'].value_counts()
client7['total_revenue'] = client.groupby('segments')['Total_spent'].sum()
client7['avg_order_value'] = client.groupby('segments')['Total_spent'].sum()/client.groupby('segments')['Completed_orders'].sum()
client7['customer_pct'] = ((client['segments'].value_counts())/client.shape[0])*100
client7

,customers,total_revenue,avg_order_value,customer_pct
segments,,,,
Regular,4933,1229888.74,38.719580,99.076120
Unknown,36,0.00,NaN,0.723037
VIP,6,504363.69,37.058317,0.120506
High Value,4,139924.14,37.233672,0.080337


In [20]:
client7[client7['total_revenue'] == client7['total_revenue'].max()]

,customers,total_revenue,avg_order_value,customer_pct
segments,,,,
Regular,4933,1229888.74,38.71958,99.07612


# 🚀 Client Requirement #8 — Restaurant Cancellation Problem

Now the operations team has a new problem.

They want to identify restaurants that have an abnormally high cancellation rate.

For each restaurant_id, calculate:

- total_orders
- cancelled_orders
- cancellation_rate
- total_revenue — Completed orders only
- avg_restaurant_rating

Then identify restaurants that satisfy both:

- At least 200 total orders
- Cancellation rate > 5%

Finally:

Give me the top 10 restaurants with the highest cancellation rate among those restaurants.

Important

For avg_restaurant_rating, use the restaurant's restaurant_rating field—not customer_rating.

In [21]:
temp = df.groupby(['restaurant_id','order_status'])[
    ['order_id',
    'order_total',
    'restaurant_rating']].agg({
        'order_id':'size',
        'order_total': 'sum',
        'restaurant_rating':'mean'
    }).unstack()
client8 = pd.DataFrame()
client8['total_orders'] = (temp['order_id']['Cancelled']) + (temp['order_id']['Completed'])
client8['cancelled_orders'] = (temp['order_id']['Cancelled'])
client8['cancellation_rate'] = (client8['cancelled_orders']/client8['total_orders'])*100
client8['total_revenue'] = temp['order_total']['Completed']
client8['avg_restaurant_rating'] = temp['restaurant_rating']['Completed']
client8


,total_orders,cancelled_orders,cancellation_rate,total_revenue,avg_restaurant_rating
restaurant_id,,,,,
RES_0001,8837.0,162.0,1.833201,286626.46,3.9
RES_0002,3553.0,52.0,1.463552,120102.18,3.6
RES_0003,3967.0,64.0,1.613310,81038.98,4.8
RES_0004,4261.0,62.0,1.455057,141270.79,4.2
RES_0005,1364.0,20.0,1.466276,44809.81,4.3
...,...,...,...,...,...
RES_0246,NaN,NaN,NaN,569.34,4.5
RES_0247,NaN,NaN,NaN,2444.45,3.3
RES_0248,NaN,NaN,NaN,426.13,4.1


In [22]:
condition = (client8['total_orders'] >= 200) & (client8['cancellation_rate']>5)

client8[condition].sort_values('cancellation_rate',ascending=False).head(10)

,total_orders,cancelled_orders,cancellation_rate,total_revenue,avg_restaurant_rating
restaurant_id,,,,,


In [23]:
client8[client8['total_orders'] >= 200].sort_values(
    'cancellation_rate',
    ascending=False
).head(10)

,total_orders,cancelled_orders,cancellation_rate,total_revenue,avg_restaurant_rating
restaurant_id,,,,,
RES_0013,690.0,24.0,3.478261,21123.15,4.8
RES_0046,272.0,8.0,2.941176,14985.86,4.4
RES_0048,254.0,7.0,2.755906,7833.89,3.9
RES_0014,439.0,12.0,2.733485,22605.37,4.2
RES_0018,753.0,19.0,2.523240,38149.93,5.0
RES_0024,242.0,6.0,2.479339,5844.57,4.4
RES_0021,569.0,14.0,2.460457,13533.26,4.2
RES_0040,306.0,7.0,2.287582,15388.24,3.6
RES_0042,291.0,6.0,2.061856,9134.42,4.4


# 🚀 Client Requirement #9 — Revenue vs Order Volume

Management wants to understand whether high-volume restaurants are also high-value restaurants.

For each restaurant_id, calculate:

- total_orders
- completed_orders
- total_revenue — Completed orders only
- avg_order_value — Completed orders only

Then create:

- revenue_per_completed_order : total_revenue / completed_orders

Then produce two rankings:

A. Top 10 restaurants by total_orders

B. Top 10 restaurants by revenue_per_completed_order

Final business question

Are the restaurants with the highest order volume also among the restaurants with the highest revenue per completed order?

In [24]:
temp = df.groupby(['restaurant_id','order_status'])[['order_id','order_total']].agg({'order_id':'size','order_total':'sum'}).unstack()
client9 = pd.DataFrame()
client9['total_orders'] = temp['order_id']['Completed'] + temp['order_id']['Cancelled']
client9['Completed_orders'] = temp['order_id']['Completed']
client9['total_revenue'] = temp['order_total']['Completed']
client9['avg_order_value'] = client9['total_revenue']/client9['Completed_orders']
client9['revenue_per_completed_order'] = client9['avg_order_value']
client9.reset_index(inplace=True)



In [25]:
(len(np.intersect1d((client9.sort_values('total_orders',ascending=False)['restaurant_id'].head(10)),(client9.sort_values('revenue_per_completed_order',ascending=False)['restaurant_id'].head(10))))/10)*100

0.0

# 🚀 Client Requirement #10 — Top Restaurants Within Each City

Management doesn't want a global restaurant ranking. They want to know the best restaurants within each city.

For every restaurant_id, calculate:

- city
- completed_orders — Completed orders only
- total_revenue — Completed orders only
- avg_order_value — Completed orders only

Then:

Within each city, rank restaurants by total_revenue, with the highest-revenue restaurant receiving rank 1.

Finally, return only the top 3 restaurants from each city.

In [26]:
temp = df.groupby(['restaurant_id','city','order_status'])[['order_id','order_total']].agg({'order_id':'size','order_total':'sum'}).unstack()

In [27]:
client10 = pd.DataFrame()
client10['Completed_orders'] = temp['order_id']['Completed']
client10['total_revenue'] = temp['order_total']['Completed']
client10['avg_order_value'] = (client10['total_revenue']/client10['Completed_orders'])
client10

,,Completed_orders,total_revenue,avg_order_value
restaurant_id,city,,,
RES_0001,City_A,8675.0,286626.46,33.040514
RES_0002,City_D,3501.0,120102.18,34.305107
RES_0003,City_C,3903.0,81038.98,20.763254
RES_0004,City_B,4199.0,141270.79,33.643913
RES_0005,City_A,1344.0,44809.81,33.340632
...,...,...,...,...
RES_0246,City_D,14.0,569.34,40.667143
RES_0247,City_B,48.0,2444.45,50.926042
RES_0248,City_D,13.0,426.13,32.779231


In [28]:
client10['Rank'] = client10.groupby('city')['total_revenue'].rank(ascending=False)

In [29]:
client10[client10['Rank'] <= 3].reset_index().set_index('city').sort_index()

,restaurant_id,Completed_orders,total_revenue,avg_order_value,Rank
city,,,,,
City_A,RES_0001,8675.0,286626.46,33.040514,1.0
City_A,RES_0005,1344.0,44809.81,33.340632,3.0
City_A,RES_0006,1127.0,58309.62,51.738793,2.0
City_B,RES_0004,4199.0,141270.79,33.643913,1.0
City_B,RES_0009,1595.0,82140.23,51.498577,2.0
City_B,RES_0018,734.0,38149.93,51.975381,3.0
City_C,RES_0003,3903.0,81038.98,20.763254,1.0
City_C,RES_0008,1247.0,62351.98,50.001588,2.0
City_C,RES_0010,1023.0,32396.16,31.667801,3.0


# 🚀 Client Requirement #11 — Customer Weekly Activity

Now we're moving into datetime analysis.

The business wants to understand how customer activity changes over time.

For each week, calculate:

- total_orders
- unique_customers
- completed_orders
- total_revenue — Completed orders only
- avg_order_value — Completed orders only

Then calculate:

- week-over-week revenue growth (%)

The business definition is:

$$ wow\ Growth = \frac{Current\ week\ Revenue - Previous\ week\ Revenue} {Previous\ week\ Revenue} \times100 $$

In [30]:
df['order_timestamp'] = pd.to_datetime(df['order_timestamp'])
df['order_date'] = pd.to_datetime(df['order_date'])

In [31]:
df['order_week'] = df['order_date'].dt.isocalendar().week

In [32]:
temp = df.groupby(['order_week','order_status'])[['customer_id','order_id','order_total']].agg({'customer_id':'nunique','order_id':'size','order_total':'sum'}).unstack()

In [33]:
client11 = pd.DataFrame()
client11['total_orders'] = temp['order_id']['Completed'] + temp['order_id']['Cancelled']
client11['unique_customer'] = df.groupby('order_week')['customer_id'].nunique()
client11['completed_orders'] = temp['order_id']['Completed']
client11['total_revenue'] = temp['order_total']['Completed']
client11['avg_order_value'] = client11['total_revenue']/client11['completed_orders'] 
client11['previous_revenue'] = client11['total_revenue'].shift(1)
client11['wow_growth'] = ((client11['total_revenue'] - client11['previous_revenue'])/client11['previous_revenue'])*100

In [34]:
client11.sort_index(ascending=False)

,total_orders,unique_customer,completed_orders,total_revenue,avg_order_value,previous_revenue,wow_growth
order_week,,,,,,,
44,6690,1668,6563,250047.08,38.099509,446534.59,-44.002752
43,11916,2323,11689,446534.59,38.201265,430917.84,3.624067
42,11487,2324,11302,430917.84,38.127574,434683.00,-0.866185
41,11620,2315,11433,434683.00,38.020030,311994.06,39.324127
40,8287,1925,8145,311994.06,38.304980,NaN,NaN


# 🚀 Client Requirement #12 — Day-of-Week Performance

The business now wants to understand which days of the week drive the most business.

For each day_of_week, calculate:

- total_orders
- unique_customers
- completed_orders
- cancelled_orders
- cancellation_rate
- total_revenue — Completed only
- avg_order_value — Completed only
- avg_delivery_time — Completed only

Then answer:

- Which 3 days generate the highest revenue?

And:

- Which day has the highest cancellation rate, provided it has at least 5,000 orders?

In [35]:
temp = df.groupby(['day_of_week','order_status'])[['order_id','order_total','actual_delivery_time_minutes']].agg({'order_id' : 'size','order_total':'sum','actual_delivery_time_minutes':'mean'}).unstack()
client12 = pd.DataFrame()
client12['total_orders'] = temp['order_id']['Completed'] + temp['order_id']['Cancelled']
client12['unique_customer'] = df.groupby('day_of_week')['customer_id'].nunique()
client12['completed_orders'] = temp['order_id']['Completed']
client12['cancelled_orders'] = temp['order_id']['Cancelled']
client12['cancellation_rate'] = (client12['cancelled_orders']/client12['total_orders'])*100
client12['total_revenue'] = temp['order_total']['Completed']
client12['avg_order_value'] = client12['total_revenue']/client12['completed_orders'] 
client12['avg_delivery_time'] = temp['actual_delivery_time_minutes']['Completed']

In [36]:
client12.sort_values('total_revenue',ascending=False).head(3)

,total_orders,unique_customer,completed_orders,cancelled_orders,cancellation_rate,total_revenue,avg_order_value,avg_delivery_time
day_of_week,,,,,,,,
Thursday,8472,1953,8316,156,1.841360,319791.44,38.454959,35.351251
Wednesday,8212,1848,8050,162,1.972723,309389.65,38.433497,35.802484
Sunday,6658,1654,6556,102,1.531992,252108.77,38.454663,33.762813


In [37]:
client12[client12['total_orders'] >= 5000].sort_values('cancellation_rate',ascending=False).head(1)

,total_orders,unique_customer,completed_orders,cancelled_orders,cancellation_rate,total_revenue,avg_order_value,avg_delivery_time
day_of_week,,,,,,,,
Wednesday,8212,1848,8050,162,1.972723,309389.65,38.433497,35.802484


# 🚀 Client Requirement #13 — Repeat Customer Analysis

Now the marketing team wants to understand customer loyalty.

A repeat customer is defined as:

- A customer who has placed at least 2 completed orders.

Using completed orders only, calculate:

- total_customers
- one_time_customers
- repeat_customers
- repeat_customer_rate
- repeat_customer_revenue
- repeat_customer_avg_order_value

Business definitions:

$$ Repeat\ Customer\ Rate = \frac{Repeat\ Customers}{Total\ Customers}\times100 $$

And:

- repeat_customer_revenue = total revenue generated by customers who have ≥2 completed orders.

- repeat_customer_avg_order_value = repeat customer revenue / completed orders made by repeat customers.

Final business question

- What percentage of completed-order revenue comes from repeat customers?₹

In [38]:
temp = df.groupby(['customer_id','order_status'])[['order_id','order_total']].agg({'order_id':'size','order_total':['sum','mean']}).unstack()

In [40]:
client13 = pd.DataFrame()
client13['completed_orders'] = temp['order_id']['size']['Completed']
client13['total_spent'] = temp['order_total']['sum']['Completed']


condition = [
    (client13['completed_orders'] >= 2),
    (client13['completed_orders'] == 1)
] 

choices = ['Repeat','one_time']

client13['customer_type'] = np.select(condition, choices, default='unknown')

repeat_customer_rate = (client13[client13['customer_type']=='Repeat'].shape[0]/client13.shape[0])*100
repeat_customer_revenue = (client13[client13['customer_type']=='Repeat']['total_spent']).sum()
repeat_customer_avg_order_value = (repeat_customer_revenue)/client13[client13['customer_type']=='Repeat']['completed_orders'].sum()
pct = ((repeat_customer_revenue)/(client13['total_spent'].sum()))*100
pct 

np.float64(95.85560446954045)

In [41]:
client13

,completed_orders,total_spent,customer_type
customer_id,,,
CUS_00001,4445.0,146326.36,Repeat
CUS_00002,2021.0,67110.38,Repeat
CUS_00003,2102.0,91792.49,Repeat
CUS_00004,2282.0,84788.20,Repeat
CUS_00005,1483.0,61700.51,Repeat
...,...,...,...
CUS_07491,1.0,15.47,one_time
CUS_07494,3.0,99.93,Repeat
CUS_07495,1.0,10.53,one_time


# 🚀 Client Requirement #14 — Discount Impact Analysis

The finance team wants to understand whether discounts are actually driving higher-value orders.

For each discount_percent level, calculate:

- total_orders
- completed_orders
- total_revenue — Completed orders only
- avg_order_value — Completed orders only
- avg_discount_percent
- avg_items_count — Completed orders only

Then create a new business metric:

Revenue per item
$$ Revenue\ Per\ Item = \frac{Total\ Revenue}{Total\ Items} $$

- Use Completed orders only for revenue and items.

Finally answer:

- Which discount level generates the highest revenue per item among discount levels with at least 500 completed orders?

# One Final Question Using two Dataset

- The analytics team wants to identify countries that experienced high COVID-19 impact during 2022.

For each country, calculate:

- Total confirmed cases in 2022
- New confirmed cases in 2022
- Total deaths in 2022
- New deaths in 2022
- 2022 case fatality rate

Use:

$$ CFR = \frac{\text{New Deaths in 2022}} {\text{New Confirmed Cases in 2022}} \times 100 $$

Important data requirements
- Aggregate Province/State rows to the country level.
- Use the cumulative nature of the dataset correctly.
- Do not simply sum cumulative case columns.
- Restrict the analysis to dates within 2022.
- Handle countries that have multiple provinces/states.
- Avoid division-by-zero problems.

Final business question:
- Among countries with at least 100,000 new confirmed cases in 2022, identify the top 10 countries with the highest 2022 case fatality rate.

In [93]:
# load the dataset

cases = pd.read_csv('datasets-session-21/time_series_covid19_confirmed_global.csv')
cases.head()

,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,12/24/22,12/25/22,12/26/22,12/27/22,12/28/22,12/29/22,12/30/22,12/31/22,1/1/23,1/2/23
0,NaN,Afghanistan,33.93911,67.709953,0,0,0,0,0,0,...,207310,207399,207438,207460,207493,207511,207550,207559,207616,207627
1,NaN,Albania,41.15330,20.168300,0,0,0,0,0,0,...,333749,333749,333751,333751,333776,333776,333806,333806,333811,333812
2,NaN,Algeria,28.03390,1.659600,0,0,0,0,0,0,...,271194,271198,271198,271202,271208,271217,271223,271228,271229,271229
3,NaN,Andorra,42.50630,1.521800,0,0,0,0,0,0,...,47686,47686,47686,47686,47751,47751,47751,47751,47751,47751
4,NaN,Angola,-11.20270,17.873900,0,0,0,0,0,0,...,104973,104973,104973,105095,105095,105095,105095,105095,105095,105095


In [94]:
# Transform dataset from Wide to Long using melt
cases = cases.melt(id_vars=['Province/State', 'Country/Region', 'Lat', 'Long'], var_name='Date', value_name='Cases')
# Converting Date column from str
cases['Date'] = pd.to_datetime(cases['Date'])
cases.head()

/var/folders/8m/g1gj7tbs71lbgrtvbdchmdy80000gn/T/ipykernel_26333/288626004.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cases['Date'] = pd.to_datetime(cases['Date'])


,Province/State,Country/Region,Lat,Long,Date,Cases
0,NaN,Afghanistan,33.93911,67.709953,2020-01-22,0
1,NaN,Albania,41.15330,20.168300,2020-01-22,0
2,NaN,Algeria,28.03390,1.659600,2020-01-22,0
3,NaN,Andorra,42.50630,1.521800,2020-01-22,0
4,NaN,Angola,-11.20270,17.873900,2020-01-22,0


In [95]:
# load the dataset

death = pd.read_csv('datasets-session-21/time_series_covid19_deaths_global.csv')
death.head()

,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,12/24/22,12/25/22,12/26/22,12/27/22,12/28/22,12/29/22,12/30/22,12/31/22,1/1/23,1/2/23
0,NaN,Afghanistan,33.93911,67.709953,0,0,0,0,0,0,...,7845,7846,7846,7846,7846,7847,7847,7849,7849,7849
1,NaN,Albania,41.15330,20.168300,0,0,0,0,0,0,...,3595,3595,3595,3595,3595,3595,3595,3595,3595,3595
2,NaN,Algeria,28.03390,1.659600,0,0,0,0,0,0,...,6881,6881,6881,6881,6881,6881,6881,6881,6881,6881
3,NaN,Andorra,42.50630,1.521800,0,0,0,0,0,0,...,165,165,165,165,165,165,165,165,165,165
4,NaN,Angola,-11.20270,17.873900,0,0,0,0,0,0,...,1928,1928,1928,1930,1930,1930,1930,1930,1930,1930


In [96]:
# Transform dataset from Wide to Long using melt
death = death.melt(id_vars=['Province/State', 'Country/Region', 'Lat', 'Long'], var_name='Date', value_name='deaths')
# Converting Date column from str
death['Date'] = pd.to_datetime(death['Date'])
death.head()

/var/folders/8m/g1gj7tbs71lbgrtvbdchmdy80000gn/T/ipykernel_26333/1138793930.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  death['Date'] = pd.to_datetime(death['Date'])


,Province/State,Country/Region,Lat,Long,Date,deaths
0,NaN,Afghanistan,33.93911,67.709953,2020-01-22,0
1,NaN,Albania,41.15330,20.168300,2020-01-22,0
2,NaN,Algeria,28.03390,1.659600,2020-01-22,0
3,NaN,Andorra,42.50630,1.521800,2020-01-22,0
4,NaN,Angola,-11.20270,17.873900,2020-01-22,0


In [97]:
cases.shape

(311253, 6)

In [98]:
death.shape

(311253, 6)

In [99]:
# Joining both the dataset
final_table = pd.merge(cases,death,on=['Province/State', 'Country/Region', 'Lat', 'Long', 'Date'])
final_table.head(5)

,Province/State,Country/Region,Lat,Long,Date,Cases,deaths
0,NaN,Afghanistan,33.93911,67.709953,2020-01-22,0,0
1,NaN,Albania,41.15330,20.168300,2020-01-22,0,0
2,NaN,Algeria,28.03390,1.659600,2020-01-22,0,0
3,NaN,Andorra,42.50630,1.521800,2020-01-22,0,0
4,NaN,Angola,-11.20270,17.873900,2020-01-22,0,0


In [100]:
# filter Dataset for year 2022 & 2021
table_2021 = final_table[final_table['Date'].dt.year == 2021]
table_2021 = table_2021.sort_values(['Country/Region','Date'])
table_2022 = final_table[final_table['Date'].dt.year == 2022]
table_2022 = table_2022.sort_values(['Country/Region','Date'])
table_2021

,Province/State,Country/Region,Lat,Long,Date,Cases,deaths
99705,NaN,Afghanistan,33.939110,67.709953,2021-01-01,52513,2201
99994,NaN,Afghanistan,33.939110,67.709953,2021-01-02,52586,2211
100283,NaN,Afghanistan,33.939110,67.709953,2021-01-03,52709,2221
100572,NaN,Afghanistan,33.939110,67.709953,2021-01-04,52909,2230
100861,NaN,Afghanistan,33.939110,67.709953,2021-01-05,53011,2237
...,...,...,...,...,...,...,...
204033,NaN,Zimbabwe,-19.015438,29.154857,2021-12-27,205449,4908
204322,NaN,Zimbabwe,-19.015438,29.154857,2021-12-28,207548,4940
204611,NaN,Zimbabwe,-19.015438,29.154857,2021-12-29,207548,4940
204900,NaN,Zimbabwe,-19.015438,29.154857,2021-12-30,211728,4997


In [110]:
table_2021 = table_2021.groupby(['Country/Region','Date'])[['Cases','deaths']].sum()
table_2021

Cases  deaths
Country/Region Date                      
Afghanistan    2021-01-01   52513    2201
               2021-01-02   52586    2211
               2021-01-03   52709    2221
               2021-01-04   52909    2230
               2021-01-05   53011    2237
...                           ...     ...
Zimbabwe       2021-12-27  205449    4908
               2021-12-28  207548    4940
               2021-12-29  207548    4940
               2021-12-30  211728    4997
               2021-12-31  213258    5004

[73365 rows x 2 columns]

In [102]:
table_2022 = table_2022.groupby(['Country/Region','Date'])[['Cases','deaths']].sum()

For each country, calculate:

- Total confirmed cases in 2022
- New confirmed cases in 2022
- Total deaths in 2022
- New deaths in 2022
- 2022 case fatality rate

In [103]:
# Total confirmed cases in 2022
# As this data has Cummulative sum in the columns
table_2022.groupby(['Country/Region']).last()['Cases']

Country/Region
Afghanistan             207559
Albania                 333806
Algeria                 271228
Andorra                  47751
Angola                  105095
                         ...  
West Bank and Gaza      703228
Winter Olympics 2022       535
Yemen                    11945
Zambia                  334425
Zimbabwe                259981
Name: Cases, Length: 201, dtype: int64

In [104]:
# New confirmed cases in 2022
# As this data has Cummulative sum in the columns using last
New_cases = table_2022.groupby(['Country/Region'])['Cases'].last() - table_2021.groupby(['Country/Region'])['Cases'].last()
New_cases

Country/Region
Afghanistan              49475
Albania                 123582
Algeria                  52796
Andorra                  24011
Angola                   23502
                         ...  
West Bank and Gaza      233480
Winter Olympics 2022       535
Yemen                     1819
Zambia                   80151
Zimbabwe                 46723
Name: Cases, Length: 201, dtype: int64

In [105]:
# Total death cases in 2022
# As this data has Cummulative sum in the columns using last
table_2022.groupby(['Country/Region'])['deaths'].last()

Country/Region
Afghanistan             7849
Albania                 3595
Algeria                 6881
Andorra                  165
Angola                  1930
                        ... 
West Bank and Gaza      5708
Winter Olympics 2022       0
Yemen                   2159
Zambia                  4024
Zimbabwe                5637
Name: deaths, Length: 201, dtype: int64

In [106]:
# New confirmed cases in 2022
# As this data has Cummulative sum in the columns using last 
New_death = table_2022.groupby(['Country/Region'])['deaths'].last() - table_2021.groupby(['Country/Region'])['deaths'].last()
New_death

Country/Region
Afghanistan             493
Albania                 378
Algeria                 605
Andorra                  25
Angola                  160
                       ... 
West Bank and Gaza      789
Winter Olympics 2022      0
Yemen                   175
Zambia                  290
Zimbabwe                633
Name: deaths, Length: 201, dtype: int64

In [107]:
# 2022 CFR

cfr = (New_death/New_cases)*100
cfr

Country/Region
Afghanistan             0.996463
Albania                 0.305870
Algeria                 1.145920
Andorra                 0.104119
Angola                  0.680793
                          ...   
West Bank and Gaza      0.337930
Winter Olympics 2022    0.000000
Yemen                   9.620671
Zambia                  0.361817
Zimbabwe                1.354793
Length: 201, dtype: float64

In [108]:
cfr[New_cases >= 100000].sort_values(ascending=False).head(10)

Country/Region
Bosnia and Herzegovina    2.539228
Egypt                     2.344122
South Africa              1.933100
North Macedonia           1.371999
Bulgaria                  1.312487
Philippines               1.137354
Colombia                  1.012449
Hungary                   1.001613
Mexico                    0.973072
Iran                      0.956949
dtype: float64

In [109]:
customer_table = pd.concat((New_cases,New_death,cfr),axis=1)
customer_table

,Cases,deaths,0
Country/Region,,,
Afghanistan,49475,493,0.996463
Albania,123582,378,0.305870
Algeria,52796,605,1.145920
Andorra,24011,25,0.104119
Angola,23502,160,0.680793
...,...,...,...
West Bank and Gaza,233480,789,0.337930
Winter Olympics 2022,535,0,0.000000
Yemen,1819,175,9.620671


# Last practice 

# Farmer Actions

In [111]:
farmer = pd.read_csv('Kaggriculture-dataset/farmer_actions.csv')
farmer.head()

,episode_id,step,day,player,action_verb,target_or_item,quantity_or_coord
0,10dfbcd2-9a84-11f1-b260-0242ac130203,0,0,0,PASS,NaN,NaN
1,10dfbcd2-9a84-11f1-b260-0242ac130203,1,0,0,BUILD_PASTURE,NaN,NaN
2,10dfbcd2-9a84-11f1-b260-0242ac130203,2,0,0,PICKUP,SHEEP,1.0
3,10dfbcd2-9a84-11f1-b260-0242ac130203,3,0,0,NORTH,NaN,NaN
4,10dfbcd2-9a84-11f1-b260-0242ac130203,4,0,0,BUILD_PASTURE,NaN,NaN


In [116]:
farmer.info()

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   episode_id         250000 non-null  str    
 1   step               250000 non-null  int64  
 2   day                250000 non-null  int64  
 3   player             250000 non-null  int64  
 4   action_verb        250000 non-null  str    
 5   target_or_item     22827 non-null   str    
 6   quantity_or_coord  12336 non-null   float64
dtypes: float64(1), int64(3), str(3)
memory usage: 13.4 MB


In [118]:
farmer.shape

(250000, 7)

In [134]:
farmer.nunique()

episode_id           186
step                 720
day                   30
player                 2
action_verb           18
target_or_item        13
quantity_or_coord     15
dtype: int64

In [138]:
farmer.isnull().sum()

episode_id                0
step                      0
day                       0
player                    0
action_verb               0
target_or_item       227173
quantity_or_coord    237664
dtype: int64

In [143]:
farmer['target_or_item'].unique()

<StringArray>
[         nan,      'SHEEP',      'WHEAT',      'MELON', 'FERTILIZER',
 'STRAWBERRY',       'WOOL',     'CARROT',       'MILK',        'COW',
      'GOOSE',     'TOMATO',        'EGG',        'ALL']
Length: 14, dtype: str

In [152]:
farmer['quantity_or_coord'].unique()

array([nan,  1.,  3.,  4.,  5.,  6.,  2., 12., 14.,  8.,  7.,  9., 10.,
        0., 11., 13.])

# Market Orders

In [112]:
market = pd.read_csv('Kaggriculture-dataset/market_orders.csv')
market.head()

,episode_id,step,day,player,order_type,item,quantity
0,10dfbcd2-9a84-11f1-b260-0242ac130203,1,0,0,HIRE,NaN,1.0
1,10dfbcd2-9a84-11f1-b260-0242ac130203,1,0,0,HIRE,NaN,1.0
2,10dfbcd2-9a84-11f1-b260-0242ac130203,1,0,0,HIRE,NaN,1.0
3,10dfbcd2-9a84-11f1-b260-0242ac130203,1,0,0,HIRE,NaN,1.0
4,10dfbcd2-9a84-11f1-b260-0242ac130203,1,0,0,HIRE,NaN,1.0


In [119]:
market.info()

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   episode_id  250000 non-null  str    
 1   step        250000 non-null  int64  
 2   day         250000 non-null  int64  
 3   player      250000 non-null  int64  
 4   order_type  250000 non-null  str    
 5   item        158784 non-null  str    
 6   quantity    250000 non-null  float64
dtypes: float64(1), int64(3), str(3)
memory usage: 13.4 MB


In [120]:
market.shape

(250000, 7)

In [132]:
market.nunique()

episode_id    168
step          719
day            30
player          2
order_type      7
item           12
quantity       87
dtype: int64

In [137]:
market.isnull().sum()

episode_id        0
step              0
day               0
player            0
order_type        0
item          91216
quantity          0
dtype: int64

In [141]:
market['item'].unique()

<StringArray>
[         nan,        'COW',      'SHEEP',      'WHEAT',      'MELON',
 'FERTILIZER', 'STRAWBERRY',     'CARROT',       'WOOL',       'MILK',
     'TOMATO',        'EGG',      'GOOSE']
Length: 13, dtype: str

# Matches 

In [113]:
matches = pd.read_csv('Kaggriculture-dataset/matches_meta.csv')
matches.head()

,episode_id,player_0_team,player_1_team,player_0_score,player_1_score,winner,margin,total_steps
0,10dfbcd2-9a84-11f1-b260-0242ac130203,Thomas Tschinkel,カワシギ,63850.0,62164.0,Player 0,1686.0,720
1,f5fc5c2a-98a5-11f1-9100-0242ac130203,vijaikm,tipstar0125,58976.0,52510.0,Player 0,6466.0,720
2,efed18c4-449c-11ef-95c7-0242ac130204,p0,p1,-1.0,1.0,Player 1,2.0,720
3,2231b716-96cd-11f1-9a77-0242ac130203,vijaikm,Bang sion,44567.0,31815.0,Player 0,12752.0,720
4,be81725a-96d7-11f1-883e-0242ac130203,Amr Yasser,vijaikm,53615.0,76927.0,Player 1,23312.0,720


In [121]:
matches.info()

<class 'pandas.DataFrame'>
RangeIndex: 537 entries, 0 to 536
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   episode_id      537 non-null    str    
 1   player_0_team   537 non-null    str    
 2   player_1_team   537 non-null    str    
 3   player_0_score  537 non-null    float64
 4   player_1_score  537 non-null    float64
 5   winner          537 non-null    str    
 6   margin          537 non-null    float64
 7   total_steps     537 non-null    int64  
dtypes: float64(3), int64(1), str(4)
memory usage: 33.7 KB


In [125]:
matches.shape

(537, 8)

In [130]:
matches.nunique()

episode_id        537
player_0_team     231
player_1_team     221
player_0_score    484
player_1_score    484
winner              3
margin            465
total_steps         1
dtype: int64

In [136]:
matches.isnull().sum()

episode_id        0
player_0_team     0
player_1_team     0
player_0_score    0
player_1_score    0
winner            0
margin            0
total_steps       0
dtype: int64

# Shop

In [114]:
shop = pd.read_csv('Kaggriculture-dataset/town_shop_schedules.csv')
shop.head()

,shop_name,product_demanded,base_multiplier,peak_multiplier
0,BAKERY,EGG,1.5,2.0
1,BAKERY,WHEAT,1.5,2.0
2,PIZZA_SHOP,MILK,1.5,2.0
3,PIZZA_SHOP,TOMATO,1.5,2.0
4,PIZZA_SHOP,WHEAT,1.5,2.0


In [123]:
shop.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   shop_name         19 non-null     str    
 1   product_demanded  19 non-null     str    
 2   base_multiplier   19 non-null     float64
 3   peak_multiplier   19 non-null     float64
dtypes: float64(2), str(2)
memory usage: 740.0 bytes


In [124]:
shop.shape

(19, 4)

In [131]:
shop.nunique()

shop_name           8
product_demanded    7
base_multiplier     1
peak_multiplier     1
dtype: int64

In [142]:
shop.isnull().sum()

shop_name           0
product_demanded    0
base_multiplier     0
peak_multiplier     0
dtype: int64